# Explainable Pothole Detection System: End-to-End Pipeline from Training to Web App Deployment

In [ ]:
!pip install --upgrade gradio uvicorn

Imports

In [ ]:
try:
  import torch
  import torchvision
  assert int(torch.__version__.split(".")[1]) >= 12
  assert int(torchvision.__version__.split(".")[1]) >= 13
  print(f"torch version: {torch.__version__}")
  print(f"torchvision version: {torchvision.__version__}")

except:
  print(f"[INFO] torch/torchvision versions not as required, installing nightly versions.")
  !pip3 install -U torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu113
  import torch
  import torchvision
  print(f"torch version: {torch.__version__}")
  print(f"torchvision version: {torchvision.__version__}")

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

try:
  from torchinfo import summary
except:
  print("[INFO] Couldn't find torchinfo... installing it.")
  !pip install -q torchinfo
  from torchinfo import summary




Setup device agnostic code

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

Get data


In [ ]:

import kagglehub

# Download latest version
path = kagglehub.dataset_download("atulyakumar98/pothole-detection-dataset")

print("Path to dataset files:", path)

In [ ]:
import shutil
from pathlib import Path

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "drive_sight_data"

if image_path.is_dir():
  shutil.rmtree(image_path)
print(f"[INFO] Copy files to {image_path}")

shutil.copytree(src=path, dst=image_path)

print("[INFO] Copying completed successfully.")


In [ ]:
import os

def walk_through_dir(dir_path):
  for dirpath, dirnames, filenames in os.walk(dir_path):
    print(f" There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

walk_through_dir(image_path)

Creating a transform

In [ ]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
weights

In [ ]:
auto_transforms = weights.transforms()
auto_transforms

In [ ]:
from torchvision import datasets, transforms
dataset = datasets.ImageFolder(root = image_path, transform = auto_transforms)

In [ ]:
from torch.utils.data import random_split

# Creating train and test data
train_size = int(0.8*len(dataset))
test_size = len(dataset)-train_size

train_data, test_data = random_split(dataset, [train_size,test_size])

print(f"Length of train data: {len(train_data)}")
print(f"Length of test data: {len(test_data)}")

Dataloaders
---



---



In [ ]:
import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()
batch_size = 32

class_names = dataset.classes
print(f"Classes: {class_names}")

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers = NUM_WORKERS,
    pin_memory=True,
)
test_dataloader = DataLoader(
    dataset=test_data,
    batch_size = batch_size,
    shuffle = False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)



Setting up a pretrained model

In [ ]:
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights = weights).to(device)

In [ ]:
# Print a summary of our model
summary(model=model,
        input_size=(32,3,224,224),
        col_names = ["input_size","output_size","num_params","trainable"],
        col_width=20,
        row_settings = ["var_names"])

Freezing the base model and changing the output layer to suit our needs

In [ ]:
effnet_b0 = torchvision.models.efficientnet_b0()
effnet_b0.classifier

In [ ]:
for param in model.parameters():
  param.requires_grad = False

In [ ]:
torch.manual_seed(42)
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=len(class_names),
                    bias=True)).to(device)

Training model

In [ ]:
import torch
from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(model:torch.nn.Module,
               dataloader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               device:torch.device) -> Tuple[float,float]:

   model.train()

   train_loss, train_acc = 0, 0

   for batch, (X,y) in enumerate(dataloader):

       X,y = X.to(device),y.to(device)
       y_pred = model(X)

       loss = loss_fn(y_pred,y)
       train_loss += loss.item()

       optimizer.zero_grad()

       loss.backward()

       optimizer.step()

       y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1),dim=1)
       train_acc += (y_pred_class == y).sum().item()/len(y_pred)

   train_loss = train_loss / len(dataloader)
   train_acc = train_acc / len(dataloader)
   return train_loss,train_acc


def test_step(model:torch.nn.Module,
              dataloader:torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              device: torch.device) -> Tuple[float,float]:

    model.eval()

    test_loss, test_acc = 0, 0

    with torch.inference_mode():
      for batch, (X,y) in enumerate(dataloader):

        X,y = X.to(device), y.to(device)

        test_pred_logits = model(X)

        loss = loss_fn(test_pred_logits, y)
        test_loss += loss.item()

        test_pred_labels = test_pred_logits.argmax(dim=1)
        test_acc += (test_pred_labels == y ).sum().item()/ len(test_pred_labels)

    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc


def train(model:torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module,
          epochs: int,
          device: torch.device) -> Dict[str, List]:

    results = {"train_loss": [],
               "train_acc": [],
               "test_loss": [],
               "test_acc": []}

    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader = train_dataloader,
                                           loss_fn = loss_fn,
                                           optimizer = optimizer,
                                           device=device)
        test_loss, test_acc = test_step(model=model,
                                       dataloader = test_dataloader,
                                       loss_fn = loss_fn,
                                       device=device)

        print(
            f"Epoch: {epoch+1} |"
            f"train_loss: {train_loss:.4f} |"
            f"train_acc: {train_acc:.4f} |"
            f"test_loss: {test_loss:.4f} |"
            f"test_acc:{test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

    return results







In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

from timeit import default_timer as timer
start_time=timer()

results = train(model = model,
                train_dataloader = train_dataloader,
                test_dataloader = test_dataloader,
                optimizer = optimizer,
                loss_fn=loss_fn,
                epochs=5,
                device=device)
end_time=timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

Creating loss curves

In [ ]:
def plot_loss_curves(results):
  loss = results["train_loss"]
  test_loss = results["test_loss"]

  accuracy = results["train_acc"]
  test_accuracy = results["test_acc"]

  epochs = range(len(results["train_loss"]))

  plt.figure(figsize=(15,7))

  plt.subplot(1,2,1)
  plt.plot(epochs,loss,label="train_loss")
  plt.plot(epochs,test_loss,label="test_loss")
  plt.title("Loss")
  plt.xlabel("Epochs")
  plt.legend()

  plt.subplot(1,2,2)
  plt.plot(epochs,accuracy,label="train_accuracy")
  plt.plot(epochs,test_accuracy,label="test_accuracy")
  plt.title("Accuracy")
  plt.xlabel("Epochs")
  plt.legend()


In [ ]:
plot_loss_curves(results)

Make predictions on images from the test set

In [ ]:
from typing import List, Tuple
from PIL import Image

def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int,int]=(224,224),
                        transform: torchvision.transforms=None,
                        device: torch.device=device):
  img = Image.open(image_path).convert("RGB")

  if transform is not None:
    image_transform=transform
  else:
    image_transform=transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485,0.456,0.406],
                             std=[0.229,0.224,0.225])
    ])

  model.to(device)

  model.eval()

  with torch.inference_mode():
    transformed_image=image_transform(img).unsqueeze(dim=0)

    target_image_pred = model(transformed_image.to(device))
  target_image_pred_probs = torch.softmax(target_image_pred, dim=1)
  target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1).item()

  plt.figure()
  plt.imshow(img)
  plt.title(f"Pred: {class_names[target_image_pred_label]} | Prob: {target_image_pred_probs.max():.3f}")
  plt.axis(False);

In [ ]:
import random
num_images_to_plot = 3
test_image_path_list = list(test_data.dataset.samples)
test_image_path_sample = random.sample(population=test_image_path_list, k=num_images_to_plot)

for image_path in test_image_path_sample:
  pred_and_plot_image(model=model,
                      image_path=image_path[0],
                      class_names=class_names,
                      transform=weights.transforms(),
                      image_size=(224,224))


Making predicitons on a custom image

In [ ]:
import requests
import os

custom_image_path = data_path / "pothole.jpeg"

# Ensure the directory exists
custom_image_path.parent.mkdir(parents=True, exist_ok=True)

# Remove existing file to force re-download, in case it was corrupted
if custom_image_path.is_file():
  os.remove(custom_image_path)
  print(f"Removed existing {custom_image_path}.")

if not custom_image_path.is_file():
  image_url = "https://ichef.bbci.co.uk/news/976/cpsprodpb/17503/production/_121519459_pothole3.jpg"
  print(f"Downloading {image_url} to {custom_image_path}...")
  request = requests.get(image_url)
  if request.status_code == 200:
    with open(custom_image_path, "wb") as f:
      f.write(request.content)
    print("Download successful.")
  else:
    print(f"Failed to download image. Status code: {request.status_code}")
    print(f"Response content type: {request.headers.get('Content-Type')}")
    # Raise an error or handle the failed download appropriately
else:
  print(f"{custom_image_path} already exists, skipping download.")

pred_and_plot_image(model=model,
                    image_path=custom_image_path,
                    class_names=class_names,
                    transform=weights.transforms(),
                    image_size=(224,224))

Creating a function to save the model

In [ ]:
import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
               target_dir: str,
               model_name: str):

  target_dir_path = Path(target_dir)
  target_dir_path.mkdir(parents=True,
                        exist_ok=True)
  assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
  model_save_path = target_dir_path / model_name

  print(f"[INFO] Saving model to: {model_save_path}")
  torch.save(obj=model.state_dict(),
             f=model_save_path)



In [ ]:
save_model(model=model,
           target_dir = "models",
           model_name = "pothole_detection_effnetb0_model.pth")

Load Model

In [ ]:
import torch
import torchvision


loaded_model = torchvision.models.efficientnet_b0()
loaded_model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features = len(class_names))
)

loaded_model.load_state_dict(torch.load("models/pothole_detection_effnetb0_model.pth"))

loaded_model = loaded_model.to(device)
loaded_model.eval()

print("[INFO] Model loaded successfully and is ready for predictions.")

In [ ]:
# Making prediction with the loaded model
pred_and_plot_image(model=loaded_model,
                    image_path=custom_image_path,
                    class_names=class_names,
                    transform=weights.transforms(),
                    image_size=(224,224))

Grad-CAM Heatmap: Visualizing the model's focus areas for decision-making

In [ ]:
!pip install captum

In [ ]:
from captum.attr import LayerGradCam
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import torch


loaded_model.eval()

target_layer = loaded_model.features[-1]

layer_gc = LayerGradCam(loaded_model, target_layer)

transformed_img = weights.transforms()(Image.open(custom_image_path)).unsqueeze(0).to(device)

attributions = layer_gc.attribute(transformed_img, target=1)

unsampled_attr = LayerGradCam.interpolate(attributions,(224,224))

heatmap = unsampled_attr.squeeze().cpu().detach().numpy()

# Convert PIL Image to PyTorch tensor before permuting
rgb_image_tensor = T.ToTensor()(T.Resize((224,224))(Image.open(custom_image_path))).permute(1,2,0)
if rgb_image_tensor.max() > 1.0:
  rgb_image_tensor = rgb_image_tensor /255.0
rgb_img = rgb_image_tensor.cpu().numpy()


plt.figure(figsize=(12,6))

plt.subplot(1,2,1)
plt.title("Original Image")
plt.imshow(rgb_img)
plt.axis(False)

plt.subplot(1,2,2)
plt.title("Grad-CAM Heatmap (Captum)")
plt.imshow(heatmap, cmap='jet', alpha=0.5)
plt.axis(False)

plt.show()

 Creating a Gradio demo

In [ ]:
try:
  import gradio as gr
except:
  !pip -q install gradio
  import gradio as gr

print(f"Gradio version: {gr.__version__}")

In [ ]:
device = torch.device("cpu")
loaded_model.to(device)
next(iter(loaded_model.parameters())).device

In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
from captum.attr import LayerGradCam
import matplotlib.pyplot as plt
import io
from PIL import Image

def predict_and_visualize(input_img):


    img_pil = input_img.convert("RGB")


    loaded_model.eval()

    transformed_img = weights.transforms()(img_pil).unsqueeze(0).to(device)

    with torch.inference_mode():
      logits = loaded_model(transformed_img)
      probs = torch.softmax(logits,dim=1).squeeze().cpu().numpy()

    pred_labels_and_probs = {class_names[i]: float(probs[i]) for i in range(len(class_names))}




    target_layer = loaded_model.features[-1]
    layer_gc = LayerGradCam(loaded_model, target_layer)

    attributions = layer_gc.attribute(transformed_img, target=1)
    unsampled_attr = LayerGradCam.interpolate(attributions, (224,224))
    heatmap = unsampled_attr.squeeze().cpu().detach().numpy()

    rgb_img = np.array(img_pil.resize((224,224)))/255.0


    fig,ax = plt.subplots(figsize=(6,6))
    ax.imshow(rgb_img)
    ax.imshow(heatmap,cmap = 'jet', alpha = 0.5)
    ax.axis(False)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches = 0)
    buf.seek(0)
    output_heatmap_img = Image.open(buf)
    plt.close(fig)

    return pred_labels_and_probs, output_heatmap_img


In [ ]:
# Create a list of example inputs to our Gradio demo

test_data_paths = list(test_data.dataset.samples)
example_list = [[str(filepath)]for filepath in random.sample(test_data_paths, k=3)]
example_list

In [ ]:
# Building a Gradio interface
import gradio as gr

title = "Explainable Pothole Detection via EfficientNet and Grad-CAM"
description = " An Explainable AI application for detecting and analyzing road damage. This tool uses a computer vision model to identify potholes and includes visual heatmaps that highlight exactly which parts of the image led to the final prediction."
gr.close_all()
demo = gr.Interface(fn = predict_and_visualize,
                    inputs= gr.Image(type="pil"),
                    outputs=[gr.Label(num_top_classes=3, label = "Prediction"),
                             gr.Image(label="Localization (Grad-CAM)")],
                    title=title,
                    description=description)

demo.launch(debug=False,
            share=True)


Turning our demo into a deployable app

In [ ]:
import shutil
from pathlib import Path

pothole_detector_demo_path = Path("demos/pothole_detector/")

if pothole_detector_demo_path.exists():
  shutil.rmtree(pothole_detector_demo_path)

pothole_detector_demo_path.mkdir(parents=True,
                                 exist_ok=True)

!ls demos/pothole_detector/

In [ ]:
import shutil
from pathlib import Path

pothole_detector_examples_path = pothole_detector_demo_path / "examples"
pothole_detector_examples_path.mkdir(parents=True,
                                     exist_ok=True)

pothole_detector_examples = [Path('data/drive_sight_data/potholes/294.jpg'),
                             Path('data/drive_sight_data/normal/62.jpg'),
                             Path('data/drive_sight_data/normal/340.jpg')]

for example in pothole_detector_examples:
  destination = pothole_detector_examples_path / example.name
  print(f"[INFO] Copying {example} to {destination}")
  shutil.copy2(src=example, dst=destination)


In [ ]:
import os

example_list = [["examples/" + example] for example in os.listdir(pothole_detector_examples_path)]
example_list

In [ ]:
import shutil

effnetb0_pothole_detector_model_path = "models/pothole_detection_effnetb0_model.pth"

effnetb0_pothole_detector_model_destination = pothole_detector_demo_path / effnetb0_pothole_detector_model_path.split("/")[1]


try:
  print(f"[INFO] Attempting to move {effnetb0_pothole_detector_model_path} to {effnetb0_pothole_detector_model_destination}")

  shutil.move(src=effnetb0_pothole_detector_model_path,
              dst=effnetb0_pothole_detector_model_destination)

  print(f"[INFO] Model move complete.")

except:
  print(f"[INFO] No model found at {effnetb0_pothole_detector_model_path}, perhaps its already been moved?")
  print(f"[INFO] Model exists at {effnetb0_pothole_detector_model_destination}: {effnetb0_pothole_detector_model_destination.exists()}")

In [ ]:
%%writefile demos/pothole_detector/app.py
import torch
import torchvision

from torch import nn
import gradio as gr
import os
import torchvision.transforms as T
import numpy as np
from captum.attr import LayerGradCam
import matplotlib.pyplot as plt
import io
from PIL import Image

device = torch.device("cpu")

class_names = ["normal", "pothole"]
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
loaded_model = torchvision.models.efficientnet_b0(weights = weights).to(device)

for param in loaded_model.parameters():
  param.requires_grad = False

torch.manual_seed(42)
loaded_model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),
    torch.nn.Linear(in_features=1280,
                    out_features=len(class_names),
                    bias=True)).to(device)


loaded_model.load_state_dict(
    torch.load(
        f="pothole_detection_effnetb0_model.pth",
        map_location=torch.device("cpu")
    )
)


def predict_and_visualize(input_img):


    img_pil = input_img.convert("RGB")


    loaded_model.eval()

    transformed_img = weights.transforms()(img_pil).unsqueeze(0).to(device)

    with torch.inference_mode():
      logits = loaded_model(transformed_img)
      probs = torch.softmax(logits,dim=1).squeeze().cpu().numpy()

    pred_labels_and_probs = {class_names[i]: float(probs[i]) for i in range(len(class_names))}




    target_layer = loaded_model.features[-1]
    layer_gc = LayerGradCam(loaded_model, target_layer)

    attributions = layer_gc.attribute(transformed_img, target=1)
    unsampled_attr = LayerGradCam.interpolate(attributions, (224,224))
    heatmap = unsampled_attr.squeeze().cpu().detach().numpy()

    rgb_img = np.array(img_pil.resize((224,224)))/255.0


    fig,ax = plt.subplots(figsize=(6,6))
    ax.imshow(rgb_img)
    ax.imshow(heatmap,cmap = 'jet', alpha = 0.5)
    ax.axis(False)

    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches = 0)
    buf.seek(0)
    output_heatmap_img = Image.open(buf)
    plt.close(fig)

    return pred_labels_and_probs, output_heatmap_img


title = "Explainable Pothole Detection via EfficientNet and Grad-CAM"
description = " An Explainable AI application for detecting and analyzing road damage. This tool uses a computer vision model to identify potholes and includes visual heatmaps that highlight exactly which parts of the image led to the final prediction."

demo = gr.Interface(fn = predict_and_visualize,
                    inputs= gr.Image(type="pil"),
                    outputs=[gr.Label(num_top_classes=3, label = "Prediction"),
                             gr.Image(label="Localization (Grad-CAM)")],
                    title=title,
                    description=description)

demo.launch()



In [ ]:
%%writefile demos/pothole_detector/requirements.txt
torch
torchvision
gradio
captum
matplotlib
numpy
Pillow

In [ ]:
!ls demos/pothole_detector

In [ ]:
!zip -r ../pothole_detector.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"

In [ ]:
!cd demos/pothole_detector && zip -r ../pothole_detector.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"


try:
    from google.colab import files
    files.download("demos/pothole_detector.zip")
except:
    print("Not running in Google Colab, can't use google.colab.files.download(), please manually download.")